In [1]:
import os
os.environ['R_HOME'] = '/home/byual/anaconda3/envs/NicheScope/lib/R'

import sys, time, importlib, pickle, json, logging, warnings, contextlib
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import style
from matplotlib.colors import ListedColormap
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp
from scipy.spatial.distance import cdist
import anndata as ad
import gseapy
import cmasher as cmr
from IPython.display import display

warnings.filterwarnings('ignore')
if 'src' not in sys.path:
    sys.path.append('src')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style
set_nature_style()

NM_W_SINGLE = 88  / 25.4
NM_W_HALF   = 120 / 25.4
NM_W_FULL   = 180 / 25.4
NM_H_MAX    = 225 / 25.4


def save_panel(fig, path, dpi=400):
    """Save as SVG (primary vector) and PNG (raster) at 400 DPI."""
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p.with_suffix('.svg'), bbox_inches='tight', facecolor='white')
    fig.savefig(p.with_suffix('.png'), dpi=dpi, bbox_inches='tight', facecolor='white')


@contextlib.contextmanager
def save_on_show(save_path, dpi_=400):
    """Monkey-patch ``plt.show`` to save SVG + PNG before displaying.
    Multiple ``plt.show`` calls are saved as ``<stem>_2``, ``<stem>_3``, ...
    """
    real_show = plt.show
    counter = [0]
    save_path_p = pathlib.Path(save_path) if save_path is not None else None

    def saving_show(*args, **kwargs):
        counter[0] += 1
        if save_path_p is not None:
            try:
                fig = plt.gcf()
                stem = (save_path_p if counter[0] == 1
                        else save_path_p.with_name(
                            f'{save_path_p.stem}_{counter[0]}{save_path_p.suffix}'))
                fig.savefig(pathlib.Path(stem).with_suffix('.svg'),
                            bbox_inches='tight', facecolor='white')
                fig.savefig(pathlib.Path(stem).with_suffix('.png'),
                            dpi=dpi_, bbox_inches='tight', facecolor='white')
            except Exception as e:
                print(f'  [save_on_show] failed to save figure: {e}')
        return real_show(*args, **kwargs)

    plt.show = saving_show
    try:
        yield
    finally:
        plt.show = real_show


In [2]:
nichescope_dir = pathlib.Path.cwd().resolve()
if not (nichescope_dir / 'src').exists():
    nichescope_fallback = pathlib.Path('/home/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope')
    if not (nichescope_fallback / 'src').exists():
        raise FileNotFoundError('Could not locate the NicheScope project directory.')
    nichescope_dir = nichescope_fallback

human_dir = nichescope_dir.parent
qc_path = human_dir / 'data' / 'qc' / 'human_merfish_qc.h5ad'
res_dir = nichescope_dir / 'results' / 'oli'
fig_dir = nichescope_dir / 'figures' / 'oli'
downstream_fig_dir = human_dir / 'Figure' / 'oli' / 'downstream'
for p in [res_dir, fig_dir, downstream_fig_dir]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
import niche_detection
importlib.reload(niche_detection)
from niche_detection import nichescope_share, nichescope_specific, compute_N

import utils
importlib.reload(utils)
from utils import (
    draw_v_radar,
    draw_u_dotmap,
    draw_niche_score_spatial,
    draw_gene_spatial,
    get_niche_cell_ids,
    draw_niche_cells,
    plot_gene_dotmap,
    plot_shared_mcn_dotmap_grid,
    plot_mcn_stgp_union_loadings,
)


## 2. Load & preprocess data

In [4]:
adata = sc.read_h5ad(qc_path)

# 定义简称到全称的映射
celltype_rename_dict = {
    'endo':   'Endothelial',
    'ext':    'Neuron-Excitatory',
    'inb':    'Neuron-Inhibitory',
    'micro':  'Microglia',
    'ast':    'Astrocyte',
    'opc':    'OPC',
    'oli':    'Oligodendrocyte',
}

# 用全称替换adata.obs["celltype"]
adata.obs['celltype'] = adata.obs['celltype'].map(celltype_rename_dict)

print(f'Full atlas: {adata.shape}')
print(f'Available ages (years): {sorted(adata.obs.age.unique())}')
print(f'Cell types: {sorted(adata.obs.celltype.unique())}')

Full atlas: (148401, 285)
Available ages (years): [15.0, 27.0, 28.0, 42.0, 49.0, 57.0, 82.0, 87.0]
Cell types: ['Astrocyte', 'Endothelial', 'Microglia', 'Neuron-Excitatory', 'Neuron-Inhibitory', 'OPC', 'Oligodendrocyte']


In [5]:
celltypes = adata.obs['celltype'].unique()

ct_cmap_dict = {
    'Endothelial':   '#A65628',   
    'Neuron-Excitatory':    '#FF7F00',   
    'Neuron-Inhibitory':    '#FFD92F', 
    'Microglia':  '#E41A1C',   
    'Astrocyte':    '#377EB8',  
    'OPC':             '#984EA3',
    'Oligodendrocyte': '#FF6500',  # vivid deep-orange for target CT
}

# niche 颜色（8 种，新类别只有 7 个，可保留原样或只取前 7 个）
niche_colors = ['#E41A1C', '#2065D6', '#882E72', '#FF7F00', '#4DAF4A', '#F781BF', '#A65628', '#FFFF33']
cmap_niche = ListedColormap(niche_colors, name='cmap_niche')

In [6]:
print(adata[adata.obs['age']==27].obs['id_region'].unique())

['5288_rep0', '5288_rep1']
Categories (2, object): ['5288_rep0', '5288_rep1']


In [7]:
AGE_YOUNG = 27.0
AGE_OLD   = 87.0
TARGET_CT = 'Oligodendrocyte'

def prep_section(adata_full, age, label):
    a = adata_full[adata_full.obs.age == age].copy()
    if age == 27:
        a = a[a.obs['id_region']=='5288_rep0'].copy()
    a.obs = a.obs.reset_index(drop=False).rename(columns={'index': 'cell_barcode'})
    a.obs['cell_id']   = np.arange(len(a.obs))
    a.obs['cell_type'] = a.obs['celltype'].astype(str)
    a.obs['x']         = a.obs['center_x'].astype(float).values
    a.obs['y']         = a.obs['center_y'].astype(float).values
    a.obs['cell_type'] = a.obs['cell_type'].astype('category')
    if not sp.issparse(a.X):
        a.X = sp.csr_matrix(a.X)
    a.layers['counts'] = a.X.copy()
    print(f'  {label}: {a.shape[0]:>6} cells, {a.shape[1]} genes, '
          f'{(a.obs.cell_type==TARGET_CT).sum():>5} {TARGET_CT}')
    return a

print(f'Subsetting young ({AGE_YOUNG} yr) and old ({AGE_OLD} yr) sections:')
adata_y = prep_section(adata, AGE_YOUNG, 'Young')
adata_o = prep_section(adata, AGE_OLD,   'Aged')

Subsetting young (27.0 yr) and old (87.0 yr) sections:


  Young:  16911 cells, 285 genes,  2097 Oligodendrocyte
  Aged:  11142 cells, 285 genes,  2970 Oligodendrocyte


In [8]:
n_genes_total = adata_y.shape[1]
n_top_genes   = n_genes_total - 5

for tag, a in [('young', adata_y), ('aged', adata_o)]:
    sc.pp.highly_variable_genes(
        a, flavor='seurat_v3', n_top_genes=n_top_genes, layer='counts'
    )
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    print(f'{tag:>5}: HVG = {int(a.var.highly_variable.sum())}, X.max = {a.X.max():.2f}')

young: HVG = 280, X.max = 8.64
 aged: HVG = 280, X.max = 8.72


In [9]:
ct_table = pd.DataFrame({
    'young': adata_y.obs.cell_type.value_counts(),
    'aged':  adata_o.obs.cell_type.value_counts(),
}).fillna(0).astype(int)
ct_table['young_pct'] = (100 * ct_table.young / ct_table.young.sum()).round(2)
ct_table['aged_pct']  = (100 * ct_table.aged  / ct_table.aged.sum()).round(2)
ct_table.sort_values('aged', ascending=False)

,young,aged,young_pct,aged_pct
cell_type,,,,
Neuron-Excitatory,7469,4213,44.17,37.81
Oligodendrocyte,2097,2970,12.40,26.66
Astrocyte,2038,1510,12.05,13.55
Neuron-Inhibitory,2588,1311,15.30,11.77
Endothelial,1450,574,8.57,5.15
OPC,599,313,3.54,2.81
Microglia,670,251,3.96,2.25


In [10]:
sc.pl.embedding(
    adata_o,
    color='CUX2',
    basis='spatial',
    vmax=10,
    cmap="viridis"  # 显式指定颜色映射表为彩色
)

### Spatial overview of the two sections

In [11]:
set_nature_style()
fig, axes = plt.subplots(1, 2, figsize=(NM_W_FULL, NM_W_FULL * 0.48),
                         gridspec_kw={'wspace': 0.06})
for ax, a, ttl in zip(axes, [adata_y, adata_o],
                      [f'Young ({int(AGE_YOUNG)} yr)', f'Aged ({int(AGE_OLD)} yr)']):
    for ct in celltypes:
        sub = a.obs.loc[a.obs.cell_type == ct]
        if len(sub) == 0:
            continue
        s     = 5   if ct == TARGET_CT else 2
        ec    = 'k' if ct == TARGET_CT else 'none'
        lw    = 0.1 if ct == TARGET_CT else 0
        alpha = 1.0 if ct == TARGET_CT else 0.55
        ax.scatter(sub.x, sub.y, s=s, c=ct_cmap_dict.get(ct, '#888888'),
                   edgecolor=ec, linewidths=lw, alpha=alpha, label=ct,
                   rasterized=True)
    ax.set_title(ttl, fontsize=11, fontweight='bold', pad=4)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for sp_ in ax.spines.values():
        sp_.set_visible(False)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left',
               fontsize=8, markerscale=2.0, frameon=False)
fig.tight_layout(pad=0.3)
save_panel(fig, fig_dir / '01_spatial_celltypes.png')
plt.show()

## 3. Shared Microglia MCNs (young + aged)

We first extract the cellular neighborhoods that **co-occur in both ages** by running NicheScope's two-condition pipeline. The Gaussian-kernel bandwidth `sigma=50` matches the median nearest-neighbor spacing in this MERFISH atlas (~165 microns at the 10th percentile, see preprocessing). We request `cca_comp=6` since 14 cell types are available.

In [12]:
SIGMA       = 50
CCA_COMP    = 3
PX, PZ      = 0.6, 0.5
USE_HVG     = n_top_genes
share_pkl   = res_dir / 'meta_share_microglia.pkl'
young_pkl   = res_dir / 'meta_specific_young.pkl'
old_pkl     = res_dir / 'meta_specific_old.pkl'

In [13]:
def load_or_compute_meta_share(force=False):
    if share_pkl.exists() and not force:
        print(f'[load] {share_pkl}')
        with open(share_pkl, 'rb') as f:
            return pickle.load(f)

    meta = nichescope_share(
        adata_y, adata_o,
        TARGET_CT,
        sigma=SIGMA,
        n_hvg=USE_HVG,
        cov_test_null=False,
        cca_comp=CCA_COMP,
        px=PX, pz=PZ,
        sort_comp_by_corr=True,
    )
    with open(share_pkl, 'wb') as f:
        pickle.dump(meta, f)
    print(f'[save] {share_pkl}')
    return meta

meta_share = load_or_compute_meta_share()

print('\nShared CCA correlations:', np.round(meta_share['cors'], 3))
print('Shared CCA scaling factors d:', np.round(meta_share['ds'], 3))

[load] /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope/results/oli/meta_share_microglia.pkl

Shared CCA correlations: [0.538 0.445 0.347]
Shared CCA scaling factors d: [6666.415 3005.59  2706.143]


In [14]:
vdf_share = meta_share['vdf'].round(3)
udf_share = meta_share['udf'].round(3)
print('Shared niche cell-type weights v:')
vdf_share

Shared niche cell-type weights v:


,comp1,comp2,comp3
Astrocyte,0.000,0.055,0.000
Endothelial,0.000,0.000,0.911
Microglia,0.000,0.000,0.000
Neuron-Excitatory,0.911,0.000,0.000
Neuron-Inhibitory,0.411,0.000,0.411
OPC,0.000,0.026,0.000
Oligodendrocyte,0.000,0.998,0.000


In [15]:
adj_ids   = list(range(meta_share['udf'].shape[1]))
adjust_u  = meta_share['udf'].iloc[:, adj_ids]
adjust_v  = meta_share['vdf'].iloc[:, adj_ids]
adjust_d  = meta_share['ds'][adj_ids]
meta_y    = nichescope_specific(
    adata_y, TARGET_CT,
    adjust_u=adjust_u, adjust_v=adjust_v, adjust_d=adjust_d,
    sigma=SIGMA, cca_comp=4,
    px=PX, pz=PZ, sort_comp_by_corr=True,
)
with open(young_pkl, 'wb') as f:
    pickle.dump(meta_y, f)

print('\nYoung-specific CCA correlations:', np.round(meta_y['cors'], 3))
meta_y['vdf'].round(3)

[2026-07-05 05:52:30] [nichescope] [INFO] Dataset summary: 16911 cells, 285 genes.


[2026-07-05 05:52:30] [nichescope] [INFO] 7 cell types: ['Astrocyte', 'Endothelial', 'Microglia', 'Neuron-Excitatory', 'Neuron-Inhibitory', 'OPC', 'Oligodendrocyte'].



[2026-07-05 05:52:31] [nichescope] [INFO] Computed neighborhood composition matrix N of shape (2097, 7).


[2026-07-05 05:52:31] [nichescope] [INFO] Nonneg CCA cors: [0.183 0.102]



[2026-07-05 05:52:31] [nichescope] [INFO] NicheScope on Oligodendrocyte: Finished in 0.9s.



Young-specific CCA correlations: [0.183 0.102]


,comp1,comp2
Astrocyte,0.984,0.000
Endothelial,0.000,0.000
Microglia,0.136,0.000
Neuron-Excitatory,0.000,0.911
Neuron-Inhibitory,0.000,0.411
OPC,0.118,0.000
Oligodendrocyte,0.000,0.000


In [16]:
adj_ids   = list(range(meta_share['udf'].shape[1]))
adjust_u  = meta_share['udf'].iloc[:, adj_ids]
adjust_v  = meta_share['vdf'].iloc[:, adj_ids]
adjust_d  = meta_share['ds'][adj_ids]
meta_o    = nichescope_specific(
    adata_o, TARGET_CT,
    adjust_u=adjust_u, adjust_v=adjust_v, adjust_d=adjust_d,
    sigma=SIGMA, cca_comp=4,
    px=PX, pz=PZ, sort_comp_by_corr=True,
)
with open(old_pkl, 'wb') as f:
    pickle.dump(meta_o, f)

print('\nAged-specific CCA correlations:', np.round(meta_o['cors'], 3))
meta_o['vdf'].round(3)

[2026-07-05 05:52:31] [nichescope] [INFO] Dataset summary: 11142 cells, 285 genes.


[2026-07-05 05:52:31] [nichescope] [INFO] 7 cell types: ['Astrocyte', 'Endothelial', 'Microglia', 'Neuron-Excitatory', 'Neuron-Inhibitory', 'OPC', 'Oligodendrocyte'].



[2026-07-05 05:52:32] [nichescope] [INFO] Computed neighborhood composition matrix N of shape (2970, 7).


[2026-07-05 05:52:32] [nichescope] [INFO] Nonneg CCA cors: [0.139 0.138 0.124 0.097]



[2026-07-05 05:52:32] [nichescope] [INFO] NicheScope on Oligodendrocyte: Finished in 0.8s.



Aged-specific CCA correlations: [0.139 0.138 0.124 0.097]


,comp1,comp2,comp3,comp4
Astrocyte,0.000,0.203,0.953,0.000
Endothelial,0.000,0.000,0.000,0.956
Microglia,0.000,0.967,0.000,0.083
Neuron-Excitatory,0.911,0.000,0.000,0.000
Neuron-Inhibitory,0.411,0.000,0.077,0.002
OPC,0.000,0.152,0.293,0.282
Oligodendrocyte,0.000,0.000,0.000,0.000


## 6. Pick informative MCNs

Heuristic: keep components whose CCA correlation is reasonably large (`>=0.05`), top dominant cell type weight `>=0.2`, and at least one gene with `u>=0.05`. This removes degenerate components.

In [17]:
meta_share['vdf']

,comp1,comp2,comp3
Astrocyte,0.000000,0.054965,0.000000
Endothelial,0.000000,0.000000,0.911438
Microglia,0.000000,0.000000,0.000000
Neuron-Excitatory,0.911438,0.000000,0.000000
Neuron-Inhibitory,0.411438,0.000000,0.411438
OPC,0.000000,0.025505,0.000000
Oligodendrocyte,0.000000,0.998162,0.000000


In [18]:
def select_meaningful_comps(meta, cor_min=0.05, v_min=0.2, u_min=0.05, exclude_ct=None):
    cors = np.asarray(meta['cors']).flatten()
    vdf  = meta['vdf']
    udf  = meta['udf']
    keep = []
    for j, comp in enumerate(vdf.columns):
        if cors[j] < cor_min:
            continue
        v = vdf[comp].copy()
        if exclude_ct is not None:
            v = v.drop(index=[c for c in exclude_ct if c in v.index])
        if v.max() < v_min:
            continue
        if udf[comp].max() < u_min:
            continue
        keep.append(comp)
    return keep

shared_comps = select_meaningful_comps(meta_share, cor_min=0.05, v_min=0.0, exclude_ct=[TARGET_CT])
young_comps  = select_meaningful_comps(meta_y,    cor_min=0.05, v_min=0.0, exclude_ct=[TARGET_CT])
old_comps    = select_meaningful_comps(meta_o,    cor_min=0.05, v_min=0.0, exclude_ct=[TARGET_CT])

# shared_comps = select_meaningful_comps(meta_share, cor_min=0.05, v_min=0.2, exclude_ct=[TARGET_CT])
# young_comps  = select_meaningful_comps(meta_y,    cor_min=0.05, v_min=0.2, exclude_ct=[TARGET_CT])
# old_comps    = select_meaningful_comps(meta_o,    cor_min=0.05, v_min=0.2, exclude_ct=[TARGET_CT])

print(f'Shared MCN components selected: {shared_comps}')
print(f'Young-specific  MCN components: {young_comps}')
print(f'Aged-specific   MCN components: {old_comps}')

Shared MCN components selected: ['comp1', 'comp2', 'comp3']
Young-specific  MCN components: ['comp1', 'comp2']
Aged-specific   MCN components: ['comp1', 'comp2', 'comp3', 'comp4']


## 7. Cell-type composition of each MCN (radar plots)

In [19]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns
_set_ns()

# Short axis labels — no collision with radar rings
_CT_ABBR = {
    'Astrocyte':          'Astrocyte',
    'Endothelial':        'Endo.',
    'Microglia':          'Microglia',
    'Neuron-Excitatory':  'Exc.\nNeuron',
    'Neuron-Inhibitory':  'Inh.\nNeuron',
    'OPC':                'OPC',
    'Oligodendrocyte':    'Oligo.',
}

def plot_radar(vdf, comps, title, palette, save_name=None):
    if len(comps) == 0:
        print(f'{title}: no components to plot.')
        return
    cts_present = [c for c in celltypes if c in vdf.index]
    vdf_plot = vdf.loc[cts_present, list(comps)]
    xticklabels = [_CT_ABBR.get(c, c) for c in cts_present]
    colors = [palette[i % len(palette)] for i in range(len(comps))]
    save_path = (fig_dir / save_name) if save_name is not None else None
    with save_on_show(save_path, dpi_=400):
        draw_v_radar(
            vdf_plot, list(comps), v_thres=0.05, colors=colors, alpha=0.55,
            width=0.36, offset=0.07, xticklabels=xticklabels, rlabel_angle=225,
            figsize=(NM_W_SINGLE * 1.12, NM_W_SINGLE * 1.12), dpi=300,
            xtick_fs=8, xlabel_pad=20,
            ylim=(0, 1.05), yticks=[0.2, 0.4, 0.6, 0.8, 1.0],
            yticklabels=['0.2', '0.4', '0.6', '0.8', '1.0'], ytick_fs=7,
            leg_loc='upper right', leg_pos=(1.42, 1.18), leg_ncol=1,
            leg_title='MCN', leg_fs=8, leg_title_fs=8,
            title=title, title_fs=10, title_y=1.10,
        )

plot_radar(meta_share['vdf'], shared_comps,
           f'Shared {TARGET_CT} MCNs', niche_colors,
           save_name='02_radar_shared.png')


In [20]:
plot_radar(meta_y['vdf'], young_comps,
           f'Young-specific {TARGET_CT} MCNs',
           ['#2E8B57', '#9ACD32', '#1B9E77', '#66A61E'],
           save_name='03_radar_young.png')

In [21]:
plot_radar(meta_o['vdf'], old_comps,
           f'Aged-specific {TARGET_CT} MCNs',
           ['#8B0000', '#B22222', '#D7301F', '#EF6548'],
           save_name='04_radar_old.png')

## 8. Spatial distribution of niche scores

For each selected MCN, we project the per-cell niche score back onto the tissue. Brighter spots indicate Microglia that participate strongly in that neighborhood.

In [22]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns

def save_png_pdf(fig, stem_path, dpi=400):
    stem_path = pathlib.Path(stem_path)
    stem_path.parent.mkdir(parents=True, exist_ok=True)
    png = stem_path.with_suffix('.png')
    pdf = stem_path.with_suffix('.pdf')
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white', pad_inches=0.04)
    fig.savefig(pdf, bbox_inches='tight', facecolor='white', pad_inches=0.04)
    print(f'Saved {png} and {pdf}')

def plot_score_pair(score_df1, score_df2, comp, label, color_hex,
                    target_ct=TARGET_CT, save_name=None, downstream_stem=None,
                    colorbar_label='Niche score'):
    _set_ns()
    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        f'cmap_{label}', ['#F4F4F4', color_hex], N=256)

    visible = [
        (sdf is not None and comp in sdf.columns)
        for sdf in [score_df1, score_df2]
    ]
    n_vis = sum(visible)
    if n_vis == 0:
        return

    # Shared colour scale across both visible panels
    global_vmax = 1e-6
    for sdf, vis in zip([score_df1, score_df2], visible):
        if vis:
            global_vmax = max(global_vmax, sdf[comp].max())

    panel_w = NM_W_FULL / 2 if n_vis == 2 else NM_W_HALF
    fig_w   = panel_w * n_vis + 0.8
    fig_h   = panel_w * 0.9
    fig, axes = plt.subplots(1, 2, figsize=(fig_w, fig_h),
                             gridspec_kw={'wspace': 0.08})

    sc_last = None
    vis_axes = []
    for ax, a, sdf, ttl, vis in zip(
        axes,
        [adata_y, adata_o],
        [score_df1, score_df2],
        [f'Young ({int(AGE_YOUNG)} yr)', f'Aged ({int(AGE_OLD)} yr)'],
        visible,
    ):
        if not vis:
            ax.set_visible(False)
            continue
        vis_axes.append(ax)
        bg = a.obs.loc[a.obs.cell_type != target_ct]
        ax.scatter(bg.x, bg.y, s=0.5, c='#EBEBEB', edgecolor='none',
                   rasterized=True)
        sdf_s = sdf.sort_values(comp).reset_index(drop=True)
        sc_h = ax.scatter(
            sdf_s.x, sdf_s.y, s=6,
            c=sdf_s[comp], cmap=cmap, vmin=0, vmax=global_vmax,
            edgecolor='none', rasterized=True,
        )
        sc_last = sc_h
        ax.set_title(ttl, fontsize=11, fontweight='bold', pad=4)
        ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])
        for sp_ in ax.spines.values():
            sp_.set_visible(False)

    # Single colorbar anchored to all visible panels
    if sc_last is not None:
        cbar = fig.colorbar(sc_last, ax=vis_axes,
                            fraction=0.028, pad=0.02,
                            shrink=0.54, aspect=16)
        cbar.set_label(colorbar_label, fontsize=9, labelpad=3)
        cbar.ax.tick_params(labelsize=8, length=2, width=0.8)
        cbar.set_ticks([0, global_vmax])
        cbar.set_ticklabels(['0', f'{global_vmax:.2f}'])
        cbar.outline.set_linewidth(0.5)
    fig.tight_layout(pad=0.3)
    if save_name is not None:
        save_panel(fig, fig_dir / save_name)
    if downstream_stem is not None:
        save_png_pdf(fig, downstream_fig_dir / downstream_stem)
    plt.show()


In [23]:
for k, comp in enumerate(shared_comps):
    top_ct = meta_share['vdf'][comp].drop(TARGET_CT, errors='ignore').idxmax()
    label = f'Shared MCN{k+1}: {TARGET_CT} | {top_ct}-rich (CCA {comp})'
    plot_score_pair(
        meta_share['score_df1'], meta_share['score_df2'],
        f'S_{comp}', label, niche_colors[k % len(niche_colors)],
        save_name=f'05_score_shared_{k+1}.png',
        downstream_stem=f'score_shared_{k+1}',
        colorbar_label=f'MCN-{k+1}\nNiche score',
    )

Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_1.png and /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_1.pdf


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_2.png and /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_2.pdf


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_3.png and /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/Figure/oli/downstream/score_shared_3.pdf


In [24]:
for k, comp in enumerate(young_comps):
    top_ct = meta_y['vdf'][comp].drop(TARGET_CT, errors='ignore').idxmax()
    label = f'Young-specific MCN{k+1}: {TARGET_CT} | {top_ct}-rich (CCA {comp})'

    plot_score_pair(
        meta_y['score_df'], None,
        f'S_{comp}', label, '#2E8B57',
        save_name=f'06_score_young_{k+1}.png',
        colorbar_label=f'Young-MCN-{k+1}\nNiche score',
    )

In [25]:
for k, comp in enumerate(old_comps):
    top_ct = meta_o['vdf'][comp].drop(TARGET_CT, errors='ignore').idxmax()
    label = f'Aged-specific MCN{k+1}: {TARGET_CT} | {top_ct}-rich (CCA {comp})'
    plot_score_pair(
        None, meta_o['score_df'],
        f'S_{comp}', label, '#B22222',
        save_name=f'07_score_old_{k+1}.png',
        colorbar_label=f'Aged-MCN-{k+1}\nNiche score',
    )

## 9. Niche-defining genes

Top-loaded genes from the sparse non-negative CCA `u` vector for each retained MCN.

In [26]:
def top_genes_table(meta, comps, n_top=20, u_min=0.05):
    udf = meta['udf']
    rows = []
    for comp in comps:
        ranked = udf[comp].sort_values(ascending=False)
        ranked = ranked[ranked >= u_min].head(n_top)
        for rank, (gene, w) in enumerate(ranked.items(), start=1):
            rows.append({'comp': comp, 'rank': rank, 'gene': gene, 'u': round(float(w), 4)})
    return pd.DataFrame(rows)

shared_gene_df = top_genes_table(meta_share, shared_comps)
young_gene_df  = top_genes_table(meta_y,    young_comps)
old_gene_df    = top_genes_table(meta_o,    old_comps)

shared_gene_df.to_csv(res_dir/'genes_shared.csv', index=False)
young_gene_df.to_csv(res_dir/'genes_young.csv', index=False)
old_gene_df.to_csv(res_dir/'genes_old.csv', index=False)

In [27]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns
import seaborn as sns
import importlib
import utils
importlib.reload(utils)
from utils import prepare_u_dotmap_frame, draw_u_dotmap_ax


def _plot_dotmap_grid(meta, comps, prefix, save_name, n_top=12):
    """All-MCN dotmaps side-by-side. No title; colourbar starts at 0."""
    if len(comps) == 0:
        print(f'{prefix}: nothing to plot.')
        return
    _set_ns()
    sns.set_theme(style='whitegrid')
    sns.set_context('paper', font_scale=0.9)

    udf = meta['udf']
    n_p = len(comps)
    mcns = ([f'{prefix}-MCN{i+1}' for i in range(n_p)] if prefix
            else [f'MCN{i+1}' for i in range(n_p)])
    cmap = sns.color_palette('vlag', as_cmap=True)

    csms = [prepare_u_dotmap_frame(udf, c, [c], n_top)[-1] for c in comps]
    global_csm = max(csms) if csms else 1.0
    if global_csm <= 0:
        global_csm = 1.0

    fig_h = max(0.36 * n_top + 1.8, 4.5)   # larger row spacing
    fig_w = min(max(2.8 * n_p + 2.8, 6.8), NM_W_FULL)
    mx = 0.5 / max(n_p - 1, 1) if n_p > 1 else 0.5
    my = 0.6 / max(n_top - 1, 1)

    fig, axes = plt.subplots(1, n_p, figsize=(fig_w, fig_h),
                             sharey=(n_p > 1), constrained_layout=False)
    if n_p == 1:
        axes = [axes]

    for k, ax in enumerate(axes):
        draw_u_dotmap_ax(
            ax, udf, comps[k], [comps[k]],
            n_top_gene=n_top, sizes=(36, 310),
            fs=9, cmap=cmap,
            xticklabels=[mcns[k]], title=None,
            mx=mx, my=my, csm=global_csm,
            hue_vmin=0,
        )

    fig.subplots_adjust(right=0.87, left=max(0.32, 1.1 / fig_w),
                        bottom=0.10, top=0.94, wspace=0.34)
    cax = fig.add_axes([0.905, 0.20, 0.016, 0.60])
    sm = mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(0, global_csm), cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_ticks([0, global_csm])
    cb.ax.tick_params(labelsize=8)
    cb.set_label('CCA gene loading (U)', fontsize=8, labelpad=4)

    save_panel(fig, fig_dir / save_name)
    print(f'Saved {fig_dir / save_name}')
    plt.show()


_plot_dotmap_grid(meta_share, shared_comps,
                  prefix='',
                  save_name='08_dotmap_shared_row.png')


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope/figures/oli/08_dotmap_shared_row.png


In [28]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns
_set_ns()

FOCUS_MCN_IDX_SHARED = 1
FOCUS_STGP_PROG = 'stGP4'
W_ACTIVE_OLI = human_dir / 'Results' / 'stgp' / 'oli' / 'W_active_genes.csv'

_ = plot_mcn_stgp_union_loadings(
    meta_share,
    shared_comps,
    W_ACTIVE_OLI,
    focus_mcn_idx=FOCUS_MCN_IDX_SHARED,
    focus_program=FOCUS_STGP_PROG,
    n_top_each_source=10,
    save_name='09_mcn_union_oli_stgp4_loadings.png',
    figure_dir=fig_dir,
)


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope/figures/oli/09_mcn_union_oli_stgp4_loadings.svg + .png


In [29]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns
import seaborn as sns
from utils import prepare_u_dotmap_frame, draw_u_dotmap_ax

_plot_dotmap_grid(meta_y, young_comps,
                  prefix='Young',
                  save_name='09_dotmap_young_all.png')


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope/figures/oli/09_dotmap_young_all.png


In [30]:
import sys as _sys
if '/home/byual/NicheScope' not in _sys.path:
    _sys.path.insert(0, '/home/byual/NicheScope')
from style import set_nature_style as _set_ns
import seaborn as sns
from utils import prepare_u_dotmap_frame, draw_u_dotmap_ax

_plot_dotmap_grid(meta_o, old_comps,
                  prefix='Aged',
                  save_name='10_dotmap_aged_all.png')


Saved /import/home4/byual/stGP-0529/RealData_HumanBrainMERFISH/NicheScope/figures/oli/10_dotmap_aged_all.png


## 11. Aging-vs-young niche gene differences

Direct gene-level comparison: difference in the union of niche-loaded genes between aged-specific and young-specific MCNs.

In [31]:
def union_top_genes(meta, comps, n_top=30, u_min=0.05):
    genes = set()
    udf = meta['udf']
    for c in comps:
        s = udf[c].sort_values(ascending=False)
        s = s[s >= u_min].head(n_top)
        genes.update(s.index.tolist())
    return genes

young_set  = union_top_genes(meta_y,    young_comps)
old_set    = union_top_genes(meta_o,    old_comps)
shared_set = union_top_genes(meta_share, shared_comps)

young_only = sorted(young_set - old_set - shared_set)
old_only   = sorted(old_set   - young_set - shared_set)
common     = list(shared_set)# sorted(young_set & old_set)

print(f'Young-only niche genes ({len(young_only)}):')
print(young_only)
print(f'\nAged-only niche genes  ({len(old_only)}):')
print(old_only)
print(f'\nCommon young+aged niche genes ({len(common)}):')
print(common)

with open(res_dir/'aging_gene_lists.json', 'w') as f:
    json.dump({'young_only': young_only, 'aged_only': old_only,
               'common_young_aged': common,
               'shared_only': sorted(shared_set - young_set - old_set)}, f, indent=2)

Young-only niche genes (2):
['COX7A2', 'SOX4']

Aged-only niche genes  (20):
['ADARB2', 'ATP5F1D', 'ATP5MC3', 'CLVS1', 'CRABP1', 'EFCAB6', 'FLT1', 'GRIK1', 'GRINA', 'LIN7B', 'MORN4', 'MYO16', 'NRG1', 'PDGFRA', 'PDZRN4', 'PTPN3', 'RHBDL1', 'SORCS3', 'SST', 'SYN3']

Common young+aged niche genes (61):
['CELF3', 'OLIG2', 'RIT2', 'SORCS2', 'HSF4', 'FAM241B', 'NPM2', 'DPF1', 'MEIS3', 'RORB', 'TMEM145', 'NUP214', 'AP1G2', 'ATOH8', 'FEZF2', 'EBF1', 'PLCH1', 'RGS11', 'APBA2', 'HES5', 'SYT5', 'AQP4', 'NEUROD6', 'RAB26', 'SLC17A7', 'ACTL6B', 'CUX2', 'PLPPR1', 'ATM', 'RGS12', 'SDHA', 'SOX13', 'LAMP5', 'TBR1', 'OLIG1', 'CHRD', 'ZMAT4', 'C1QL3', 'CBLN2', 'NKX2-2', 'GRM1', 'PVALB', 'RNF208', 'PDIA2', 'SOX2', 'EPHB1', 'CORO6', 'MAD2L2', 'KCNB2', 'CLDN5', 'TBX2', 'SLC1A2', 'MAT2A', 'SATB2', 'SYNPR', 'RAPGEFL1', 'LMO2', 'ID2', 'ASCL1', 'MOG', 'SV2C']


In [32]:
from scipy.stats import ranksums

micro_y = adata_y[adata_y.obs.cell_type == TARGET_CT].copy()
micro_o = adata_o[adata_o.obs.cell_type == TARGET_CT].copy()

common_genes = sorted(set(micro_y.var_names) & set(micro_o.var_names))
X_y = micro_y[:, common_genes].X.toarray()
X_o = micro_o[:, common_genes].X.toarray()

stats, pvals = [], []
for j in range(len(common_genes)):
    s, p = ranksums(X_o[:, j], X_y[:, j])
    stats.append(s); pvals.append(p)

de_df = pd.DataFrame({
    'gene': common_genes,
    'mean_young': X_y.mean(0),
    'mean_aged':  X_o.mean(0),
    'log2fc_aged_vs_young': np.log2((X_o.mean(0) + 1e-3) / (X_y.mean(0) + 1e-3)),
    'wilcoxon_z': stats,
    'pval': pvals,
}).sort_values('pval')
de_df['adj_p'] = np.minimum(de_df['pval'] * len(de_df), 1.0)
de_df.to_csv(res_dir/'microglia_DE_aged_vs_young.csv', index=False)
print('Top 20 ext genes upregulated in aged vs young:')
de_df.sort_values('log2fc_aged_vs_young', ascending=False).head(20)

Top 20 ext genes upregulated in aged vs young:


,gene,mean_young,mean_aged,log2fc_aged_vs_young,wilcoxon_z,pval,adj_p
104,FKBP5,1.162028,3.978229,1.774603,42.042141,0.000000e+00,0.000000e+00
186,PDIA2,0.916014,2.886645,1.654877,24.544091,5.001283e-133,1.425366e-130
243,SORCS2,2.484694,4.645703,0.902559,38.566146,0.000000e+00,0.000000e+00
15,ATOH8,0.332822,0.567787,0.768810,3.448455,5.638044e-04,1.606843e-01
71,CNTN5,0.183347,0.310559,0.757082,1.824965,6.800634e-02,1.000000e+00
167,NEIL1,0.602692,1.001648,0.731930,5.950105,2.679707e-09,7.637165e-07
7,AP1G2,0.256551,0.413530,0.686615,2.091223,3.650811e-02,1.000000e+00
40,Blank-29,0.071250,0.114733,0.679728,0.622539,5.335875e-01,1.000000e+00
37,Blank-26,0.064306,0.096268,0.574749,0.445563,6.559126e-01,1.000000e+00
283,ZNF804B,0.135990,0.201069,0.560779,0.898517,3.689101e-01,1.000000e+00


In [33]:
from style import set_nature_style
set_nature_style()
from adjustText import adjust_text as _adj

def _gene_mcn_map(meta, comps, prefix, n_top=30, u_min=0.05):
    mapping = {}
    for k, c in enumerate(comps):
        lbl = f'{prefix}-MCN-{k+1}' if prefix else f'MCN-{k+1}'
        s = meta['udf'][c].sort_values(ascending=False)
        for g in s[s >= u_min].head(n_top).index:
            mapping.setdefault(g, []).append(lbl)
    return mapping

mcn_common = _gene_mcn_map(meta_share, shared_comps, '')
mcn_young  = _gene_mcn_map(meta_y,     young_comps,  'Young')
mcn_aged   = _gene_mcn_map(meta_o,     old_comps,    'Aged')

x = de_df['log2fc_aged_vs_young'].values
y = -np.log10(np.clip(de_df['adj_p'].values, 1e-200, 1))

is_aged   = de_df['gene'].isin(old_only).values
is_young  = de_df['gene'].isin(young_only).values
is_common = de_df['gene'].isin(common).values
is_other  = ~(is_aged | is_young | is_common)

_COL   = {'other': '#CCCCCC', 'common': '#4472C4', 'young': '#2E8B57', 'aged': '#C0392B'}
_ALPHA = {'other': 0.50, 'common': 0.88, 'young': 0.88, 'aged': 0.88}
_SIZE  = {'other': 6,   'common': 28,   'young': 28,   'aged': 28}
_LBL   = {
    'other':  'Other genes',
    'common': 'Shared MCN genes',
    'young':  'Young-specific MCN genes',
    'aged':   'Aged-specific MCN genes',
}

fig, ax = plt.subplots(figsize=(NM_W_FULL, 4.8))

for mask, key in [(is_other, 'other'), (is_common, 'common'),
                  (is_young, 'young'), (is_aged, 'aged')]:
    ax.scatter(x[mask], y[mask],
               s=_SIZE[key], c=_COL[key], alpha=_ALPHA[key],
               edgecolors='none' if key == 'other' else 'k',
               linewidths=0 if key == 'other' else 0.3,
               label=_LBL[key], rasterized=(key == 'other'), zorder=3)

fdr_y = -np.log10(0.05)
ax.axhline(fdr_y, ls='--', c='#555555', lw=0.8, zorder=2)
ax.text(x.max() * 1.01, fdr_y + 0.12, 'FDR = 0.05',
        fontsize=8, color='#555555', ha='right', va='bottom')
ax.axvline(0, ls='--', c='#555555', lw=0.8, zorder=2)

texts = []
for ann_genes, mcn_dict, col in [
    (old_only,   mcn_aged,   _COL['aged']),
    (young_only, mcn_young,  _COL['young']),
    (common,     mcn_common, _COL['common']),
]:
    for g in ann_genes:
        if g not in de_df['gene'].values:
            continue
        row = de_df.set_index('gene').loc[g]
        if float(row['adj_p']) > 1e-25:
            continue
        gx = float(row['log2fc_aged_vs_young'])
        gy = float(-np.log10(max(row['adj_p'], 1e-200)))
        mcn_str = '/'.join(mcn_dict.get(g, []))
        ann_lbl = f'{g}\n({mcn_str})' if mcn_str else g
        t = ax.text(gx, gy, ann_lbl, fontsize=6.5, color=col,
                    fontstyle='italic', linespacing=1.2,
                    ha='center', va='bottom')
        texts.append(t)

if texts:
    _adj(texts, ax=ax,
         arrowprops=dict(arrowstyle='-', color='#888888', lw=0.45),
         expand=(1.15, 1.35), force_text=(0.30, 0.50),
         only_move={'points': 'y', 'text': 'xy'}, time_lim=5)

ax.set_xlabel(r'$\log_2$ fold-change (aged / young) in Oligodendrocyte',
              fontsize=11)
ax.set_ylabel(r'$-\log_{10}$ adj. $p$-value', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

leg = ax.legend(frameon=False, fontsize=9,
                loc='upper left', bbox_to_anchor=(0.01, 0.99))
for lh in leg.legend_handles:
    lh.set_alpha(1.0)
    try:
        lh.set_sizes([28])
    except Exception:
        pass

fig.tight_layout(pad=0.4)
save_panel(fig, fig_dir / '14_ext_volcano.png')
plt.show()
